# GSE320042 Visium Merge To AnnData

This notebook discovers downloaded GSE320042 Visium files, loads each sample as `AnnData`, and concatenates everything into one merged object.


In [3]:
from __future__ import annotations

import os
from pathlib import Path
import re

os.environ.setdefault("NUMBA_CACHE_DIR", str(Path(".numba_cache").resolve()))

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc


In [4]:
# Resolve project root robustly even if notebook is launched from notebooks/
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
ROOT = None
for c in candidates:
    if (c / 'data').exists() and (c / 'notebooks').exists():
        ROOT = c
        break
if ROOT is None:
    raise RuntimeError(f'Could not locate project root from {cwd}')

DATA_DIR = ROOT / 'data' / 'GSE320042_visium' / 'files'
OUT_PATH = ROOT / 'data' / 'GSE320042_visium' / 'GSE320042_visium_merged.h5ad'

assert DATA_DIR.exists(), f'Missing data dir: {DATA_DIR}'
print('ROOT:', ROOT)
print('DATA_DIR:', DATA_DIR)


ROOT: /Users/chrislangseth/work/karolinska_institutet/projects/KaroSpaceDataWrangling
DATA_DIR: /Users/chrislangseth/work/karolinska_institutet/projects/KaroSpaceDataWrangling/data/GSE320042_visium/files


In [5]:
h5_files = sorted(DATA_DIR.glob("*_filtered_feature_bc_matrix.h5"))
print(f"Found {len(h5_files)} matrix files")
for p in h5_files[:10]:
    print(" -", p.name)


Found 17 matrix files
 - GSM9532655_WU1340_filtered_feature_bc_matrix.h5
 - GSM9532656_WU1373_filtered_feature_bc_matrix.h5
 - GSM9532657_WU1384_1_filtered_feature_bc_matrix.h5
 - GSM9532658_WU1384_2_filtered_feature_bc_matrix.h5
 - GSM9532659_WU1457_HD_016um_filtered_feature_bc_matrix.h5
 - GSM9532660_WU1609_filtered_feature_bc_matrix.h5
 - GSM9532661_WU2109_HD_016um_filtered_feature_bc_matrix.h5
 - GSM9532662_WU2130_1_filtered_feature_bc_matrix.h5
 - GSM9532663_WU2130_2_filtered_feature_bc_matrix.h5
 - GSM9532664_WU2415_filtered_feature_bc_matrix.h5


In [6]:
def _sample_id_from_h5(path: Path) -> str:
    return path.name.replace("_filtered_feature_bc_matrix.h5", "")


def _positions_path(sample_id: str) -> Path | None:
    csv_path = DATA_DIR / f"{sample_id}_tissue_positions.csv.gz"
    if csv_path.exists():
        return csv_path
    parquet_path = DATA_DIR / f"{sample_id}_tissue_positions.parquet.gz"
    if parquet_path.exists():
        return parquet_path
    return None


def _read_positions(path: Path) -> pd.DataFrame:
    if path.suffixes[-2:] == [".csv", ".gz"]:
        df = pd.read_csv(path, header=None)
        # 10x format: barcode,in_tissue,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres
        if df.shape[1] >= 6:
            df = df.iloc[:, :6].copy()
            df.columns = [
                "barcode",
                "in_tissue",
                "array_row",
                "array_col",
                "pxl_row_in_fullres",
                "pxl_col_in_fullres",
            ]
        else:
            raise ValueError(f"Unexpected tissue_positions CSV format: {path}")
        return df

    if path.suffixes[-2:] == [".parquet", ".gz"]:
        df = pd.read_parquet(path)
        # Normalize likely Visium HD parquet schema
        rename_map = {}
        if "barcode" not in df.columns:
            for c in df.columns:
                if c.lower() == "barcode":
                    rename_map[c] = "barcode"
        if "pxl_row_in_fullres" not in df.columns:
            for c in df.columns:
                if c.lower() in {"pxl_row_in_fullres", "y", "pixel_y", "imagey"}:
                    rename_map[c] = "pxl_row_in_fullres"
                    break
        if "pxl_col_in_fullres" not in df.columns:
            for c in df.columns:
                if c.lower() in {"pxl_col_in_fullres", "x", "pixel_x", "imagex"}:
                    rename_map[c] = "pxl_col_in_fullres"
                    break
        if rename_map:
            df = df.rename(columns=rename_map)
        return df

    raise ValueError(f"Unsupported positions file: {path}")


def _extract_meta(sample_id: str) -> dict:
    # Example: GSM9532659_WU1457_HD_016um
    m = re.match(r"^(GSM\d+)_(.+)$", sample_id)
    gsm = m.group(1) if m else None
    specimen = m.group(2) if m else sample_id
    is_hd = "_HD_" in sample_id
    return {
        "sample_id": sample_id,
        "gsm": gsm,
        "specimen": specimen,
        "platform": "Visium HD" if is_hd else "Visium",
    }


def load_one_sample(h5_path: Path) -> ad.AnnData:
    sample_id = _sample_id_from_h5(h5_path)
    meta = _extract_meta(sample_id)

    a = sc.read_10x_h5(h5_path)
    a.var_names_make_unique()

    for k, v in meta.items():
        a.obs[k] = v

    pos_path = _positions_path(sample_id)
    if pos_path is not None:
        pos = _read_positions(pos_path)
        if "barcode" in pos.columns:
            pos = pos.set_index("barcode")
            # Match barcodes robustly for cases with/without -1 suffix
            if not pos.index.isin(a.obs_names).any():
                pos.index = pos.index.astype(str) + "-1"
            pos = pos.reindex(a.obs_names)

            if "in_tissue" in pos.columns:
                a.obs["in_tissue"] = pos["in_tissue"].astype("float")
            if "array_row" in pos.columns:
                a.obs["array_row"] = pos["array_row"]
            if "array_col" in pos.columns:
                a.obs["array_col"] = pos["array_col"]

            if {"pxl_col_in_fullres", "pxl_row_in_fullres"}.issubset(pos.columns):
                a.obsm["spatial"] = pos[["pxl_col_in_fullres", "pxl_row_in_fullres"]].to_numpy()

    return a


In [7]:
adatas = []
for h5 in h5_files:
    try:
        a = load_one_sample(h5)
        adatas.append(a)
        print(f"Loaded {h5.name}: {a.n_obs} spots/bins x {a.n_vars} genes")
    except Exception as e:
        print(f"FAILED {h5.name}: {e}")

print(f"\nSuccessfully loaded {len(adatas)} / {len(h5_files)} samples")


/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Loaded GSM9532655_WU1340_filtered_feature_bc_matrix.h5: 4991 spots/bins x 18085 genes


/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Loaded GSM9532656_WU1373_filtered_feature_bc_matrix.h5: 4708 spots/bins x 18085 genes


/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


...

/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/chrislangseth/miniforge3/

Loaded GSM9532668_YUALT_filtered_feature_bc_matrix.h5: 2558 spots/bins x 18085 genes
Loaded GSM9532669_YUBOISE_filtered_feature_bc_matrix.h5: 476 spots/bins x 18085 genes


/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Loaded GSM9532670_YUMAZO_filtered_feature_bc_matrix.h5: 2597 spots/bins x 18085 genes
Loaded GSM9532671_YUSTE_filtered_feature_bc_matrix.h5: 2785 spots/bins x 18085 genes

Successfully loaded 15 / 17 samples


/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [8]:
if not adatas:
    raise RuntimeError("No samples loaded; cannot merge")

merged = ad.concat(
    adatas,
    axis=0,
    join="outer",
    label="sample_batch",
    keys=[a.obs["sample_id"].iloc[0] for a in adatas],
    index_unique="__",
    merge="same",
)

print(merged)
print(merged.obs[["sample_id", "gsm", "specimen", "platform"]].drop_duplicates().head(20))


AnnData object with n_obs × n_vars = 56169 × 18085
    obs: 'sample_id', 'gsm', 'specimen', 'platform', 'in_tissue', 'array_row', 'array_col', 'sample_batch'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'
                                                   sample_id         gsm  \
AACACCTACTATCGAA-1__GSM9532655_WU1340      GSM9532655_WU1340  GSM9532655   
AACACCTACTATCGAA-1__GSM9532656_WU1373      GSM9532656_WU1373  GSM9532656   
AACACGTGCATCGCAC-1__GSM9532657_WU1384_1  GSM9532657_WU1384_1  GSM9532657   
AACACCTACTATCGAA-1__GSM9532658_WU1384_2  GSM9532658_WU1384_2  GSM9532658   
AACACCTACTATCGAA-1__GSM9532660_WU1609      GSM9532660_WU1609  GSM9532660   
AACACCTACTATCGAA-1__GSM9532662_WU2130_1  GSM9532662_WU2130_1  GSM9532662   
AACACCTACTATCGAA-1__GSM9532663_WU2130_2  GSM9532663_WU2130_2  GSM9532663   
AACACCTACTATCGAA-1__GSM9532664_WU2415      GSM9532664_WU2415  GSM9532664   
AACACGTGCATCGCAC-1__GSM9532665_WU3049      GSM9532665_WU3049  GSM9532665   
AACACCTACTATCGA

In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
merged.write_h5ad(OUT_PATH, compression="gzip")
print(f"Saved merged AnnData to: {OUT_PATH}")


In [ ]:
# Optional quick QC summary
summary = merged.obs.groupby(["platform", "sample_id"]).size().rename("n_obs").reset_index()
summary.sort_values("sample_id").head(30)


## Optional: Convert Seurat RDS To H5AD

Use this section if you want to preserve Seurat-level metadata/reductions from `*_visium_seuratobj.rds`.


In [ ]:
import subprocess

RDS2H5AD_REPO = Path('/Users/chrislangseth/work/karolinska_institutet/projects/RDStoH5AD')
RDS_OUTPUT_DIR = DATA_DIR / 'converted_h5ad'
RDS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rds_files = sorted(DATA_DIR.glob('*_visium_seuratobj.rds')) + sorted(DATA_DIR.glob('*_seuratobj.rds'))
print(f'Found {len(rds_files)} RDS files')
for r in rds_files[:10]:
    print(' -', r.name)


In [ ]:
# Convert each RDS -> H5AD via the local converter.
# Requires the converter env/R deps to be installed.
converted = []
for rds in rds_files:
    out = RDS_OUTPUT_DIR / (rds.stem + '.h5ad')
    cmd = [
        'python', '-m', 'rdstoh5ad.cli', 'convert',
        str(rds), str(out),
    ]
    print('Converting', rds.name)
    res = subprocess.run(cmd, cwd=RDS2H5AD_REPO, capture_output=True, text=True)
    if res.returncode != 0:
        print('FAILED:', rds.name)
        print(res.stderr[-1000:])
        continue
    converted.append(out)

print(f'Converted {len(converted)} / {len(rds_files)}')


In [ ]:
# Optional merge of converted Seurat-derived h5ad files
converted_h5ad = sorted(RDS_OUTPUT_DIR.glob('*.h5ad'))
if converted_h5ad:
    converted_adatas = [ad.read_h5ad(x) for x in converted_h5ad]
    merged_from_rds = ad.concat(converted_adatas, axis=0, join='outer', merge='same', index_unique='__')
    out_rds_merged = OUT_PATH.with_name('GSE320042_visium_merged_from_rds.h5ad')
    merged_from_rds.write_h5ad(out_rds_merged, compression='gzip')
    print('Saved', out_rds_merged)
else:
    print('No converted h5ad files found.')
